In [1]:
# ==========================================
# Feature Engineering
# Lahore AQI Prediction Project
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
data_path = Path(
    "../data/processed/lahore/lahore_cleaned.csv"
)

lahore_df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", lahore_df.shape)

Dataset loaded successfully.
Shape: (22632, 15)


In [3]:
lahore_df["datetime"] = pd.to_datetime(
    lahore_df["datetime"]
)

print(lahore_df["datetime"].dtype)

datetime64[us]


In [4]:
lahore_df = (
    lahore_df
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Data sorted chronologically.")

Data sorted chronologically.


In [5]:
is_sorted = lahore_df["datetime"].is_monotonic_increasing

print("Datetime sorted:", is_sorted)

Datetime sorted: True


In [7]:
# ==========================================
# Time-based features
# ==========================================

lahore_df["hour"] = lahore_df["datetime"].dt.hour

lahore_df["day"] = lahore_df["datetime"].dt.day

lahore_df["month"] = lahore_df["datetime"].dt.month

lahore_df["day_of_week"] = (
    lahore_df["datetime"].dt.dayofweek
)

lahore_df["is_weekend"] = (
    lahore_df["day_of_week"] >= 5
).astype(int)

In [8]:
lahore_df[
    [
        "datetime",
        "hour",
        "day",
        "month",
        "day_of_week",
        "is_weekend"
    ]
].head(10)

,datetime,hour,day,month,day_of_week,is_weekend
0,2024-01-01 00:00:00,0,1,1,0,0
1,2024-01-01 01:00:00,1,1,1,0,0
2,2024-01-01 02:00:00,2,1,1,0,0
3,2024-01-01 03:00:00,3,1,1,0,0
4,2024-01-01 04:00:00,4,1,1,0,0
5,2024-01-01 05:00:00,5,1,1,0,0
6,2024-01-01 06:00:00,6,1,1,0,0
7,2024-01-01 07:00:00,7,1,1,0,0
8,2024-01-01 08:00:00,8,1,1,0,0
9,2024-01-01 09:00:00,9,1,1,0,0


In [9]:
# ==========================================
# Cyclical time encoding
# ==========================================

lahore_df["hour_sin"] = np.sin(
    2 * np.pi * lahore_df["hour"] / 24
)

lahore_df["hour_cos"] = np.cos(
    2 * np.pi * lahore_df["hour"] / 24
)

lahore_df["month_sin"] = np.sin(
    2 * np.pi * lahore_df["month"] / 12
)

lahore_df["month_cos"] = np.cos(
    2 * np.pi * lahore_df["month"] / 12
)

In [10]:
lahore_df["aqi_change_rate"] = (
    lahore_df["us_aqi"].diff()
)

In [11]:
lahore_df["aqi_lag_1h"] = (
    lahore_df["us_aqi"].shift(1)
)

In [12]:
lahore_df["aqi_lag_24h"] = (
    lahore_df["us_aqi"].shift(24)
)

In [13]:
lahore_df["aqi_rolling_mean_6h"] = (
    lahore_df["us_aqi"]
    .rolling(window=6)
    .mean()
)

In [14]:
lahore_df["aqi_rolling_std_6h"] = (
    lahore_df["us_aqi"]
    .rolling(window=6)
    .std()
)

In [15]:
lahore_df["pm25_rolling_mean_6h"] = (
    lahore_df["pm2_5"]
    .rolling(window=6)
    .mean()
)

In [16]:
print("Dataset shape:", lahore_df.shape)

print("\nColumns:")
for column in lahore_df.columns:
    print(column)

Dataset shape: (22632, 30)

Columns:
datetime
temperature_2m
relative_humidity_2m
surface_pressure
precipitation
cloud_cover
wind_speed_10m
wind_direction_10m
pm2_5
pm10
carbon_monoxide
nitrogen_dioxide
sulphur_dioxide
ozone
us_aqi
hour
day
month
day_of_week
is_weekend
hour_sin
hour_cos
month_sin
month_cos
aqi_change_rate
aqi_lag_1h
aqi_lag_24h
aqi_rolling_mean_6h
aqi_rolling_std_6h
pm25_rolling_mean_6h


In [17]:
print("\nMissing values:")
print(lahore_df.isnull().sum())


Missing values:
datetime                 0
temperature_2m           0
relative_humidity_2m     0
surface_pressure         0
precipitation            0
cloud_cover              0
wind_speed_10m           0
wind_direction_10m       0
pm2_5                    0
pm10                     0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
us_aqi                   0
hour                     0
day                      0
month                    0
day_of_week              0
is_weekend               0
hour_sin                 0
hour_cos                 0
month_sin                0
month_cos                0
aqi_change_rate          1
aqi_lag_1h               1
aqi_lag_24h             24
aqi_rolling_mean_6h      5
aqi_rolling_std_6h       5
pm25_rolling_mean_6h     5
dtype: int64


In [18]:
lahore_df = (
    lahore_df
    .dropna()
    .reset_index(drop=True)
)

In [19]:
print("Final shape:", lahore_df.shape)

print("\nMissing values after feature engineering:")
print(lahore_df.isnull().sum().sum())

Final shape: (22608, 30)

Missing values after feature engineering:
0


In [20]:
time_diff = (
    lahore_df["datetime"]
    .diff()
    .dropna()
)

print(
    "Non-hourly intervals:",
    (time_diff != pd.Timedelta(hours=1)).sum()
)

Non-hourly intervals: 0


In [21]:
FEATURE_OUTPUT_PATH = Path(
    "../data/processed/lahore/lahore_features_hourly.csv"
)

FEATURE_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

lahore_df.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False
)

print("Feature dataset saved successfully.")
print(FEATURE_OUTPUT_PATH)

Feature dataset saved successfully.
..\data\processed\lahore\lahore_features_hourly.csv


In [22]:
print("=" * 50)
print("FEATURE DATASET VALIDATION")
print("=" * 50)

print("Shape:", lahore_df.shape)

print(
    "Start:",
    lahore_df["datetime"].min()
)

print(
    "End:",
    lahore_df["datetime"].max()
)

print(
    "Duplicate timestamps:",
    lahore_df["datetime"].duplicated().sum()
)

print(
    "Missing values:",
    lahore_df.isnull().sum().sum()
)

print(
    "Non-hourly intervals:",
    (
        lahore_df["datetime"].diff().dropna()
        != pd.Timedelta(hours=1)
    ).sum()
)

print("=" * 50)

FEATURE DATASET VALIDATION
Shape: (22608, 30)
Start: 2024-01-02 00:00:00
End: 2026-07-31 23:00:00
Duplicate timestamps: 0
Missing values: 0
Non-hourly intervals: 0
